# Demo: Understanding Context from Music

A short, end-to-end walkthrough using the checkpoints already trained by `scripts/run_task1.py` .. `run_task4.py`:

1. **Task 1** -- predict tags for one caption
2. **Task 2** -- predict genre for one GTZAN track's graph
3. **Task 3** -- predict tags from caption + audio graph together (concat fusion)
4. **Task 4** -- retrieve the top-3 audio clips for one caption

Run `scripts/run_task1.py` through `run_task4.py` first so the checkpoints in `results/checkpoints/` exist.

In [ ]:
import os, sys, json

# Jupyter starts in this notebooks/ folder -- find the project root (one
# level up) and add it (plus its src/ folder) to the path, then move there
# so all the relative paths in config.py (data/..., results/...) work too.
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
os.chdir(PROJECT_ROOT)

import torch
import config
from common import get_device

device = get_device()
print("Using device:", device)

## 1. Task 1 -- caption to tags

In [ ]:
from bert_encoder import BertTagClassifier

vocab1 = json.loads((config.SPLITS_DIR / "tag_vocab_task1.json").read_text())
test1 = json.loads((config.SPLITS_DIR / "task1_test.json").read_text())

model1 = BertTagClassifier(config.TASK1_MODEL_NAME, num_labels=len(vocab1)).to(device)
model1.load_state_dict(torch.load(config.CHECKPOINT_DIR / "task1_bert_best.pt", map_location=device))
model1.eval()

example = test1[0]
batch = model1.encoder.tokenize([example["text"]], max_length=config.TASK1_MAX_TEXT_LEN)
with torch.no_grad():
    probs = torch.sigmoid(model1(batch["input_ids"].to(device), batch["attention_mask"].to(device)))[0]
predicted = [vocab1[i] for i in (probs > 0.5).nonzero().flatten().tolist()]

print("Caption:", example["text"])
print("True tags:", example["tags"])
print("Predicted tags:", predicted)

## 2. Task 2 -- audio graph to genre

In [ ]:
from gnn_model import GNNGenreClassifier
from torch_geometric.data import Batch

genre_vocab = json.loads((config.SPLITS_DIR / "gtzan_genre_vocab.json").read_text())
test2 = json.loads((config.SPLITS_DIR / "gtzan_test.json").read_text())

model2 = GNNGenreClassifier(
    in_channels=config.GRAPH_IN_CHANNELS, num_classes=len(genre_vocab),
    hidden_channels=config.TASK2_GNN_HIDDEN, out_channels=config.TASK2_GNN_OUT,
    num_layers=config.TASK2_GNN_LAYERS, arch=config.TASK2_GNN_ARCH,
).to(device)
model2.load_state_dict(torch.load(config.CHECKPOINT_DIR / "task2_gnn_best.pt", map_location=device))
model2.eval()

example2 = test2[0]
graph = torch.load(example2["graph_path"], weights_only=False)
batch = Batch.from_data_list([graph]).to(device)
with torch.no_grad():
    pred_idx = model2(batch.x, batch.edge_index, batch.batch).argmax(dim=1).item()

print("Track:", example2["track_id"], " nodes in graph:", graph.num_nodes)
print("True genre:", example2["genre"])
print("Predicted genre:", genre_vocab[pred_idx])

## 3. Task 3 -- caption + audio graph to tags (concat fusion)

In [ ]:
from task3_fusion import GNNBertFusion
from torch_geometric.data import Batch as PyGBatch

vocab34 = json.loads((config.SPLITS_DIR / "tag_vocab_task34.json").read_text())
test34 = json.loads((config.SPLITS_DIR / "task34_test.json").read_text())

model3 = GNNBertFusion(
    bert_model_name=config.TASK1_MODEL_NAME, gnn_in_channels=config.GRAPH_IN_CHANNELS,
    num_labels=len(vocab34), attn_dim=config.TASK3_ATTN_DIM, mode="concat",
).to(device)
model3.load_state_dict(torch.load(config.CHECKPOINT_DIR / "task3_fusion_concat_best.pt", map_location=device))
model3.eval()

example3 = test34[0]
graph3 = torch.load(example3["graph_path"], weights_only=False)
graph_batch = PyGBatch.from_data_list([graph3]).to(device)
text_batch = model3.bert.tokenize([example3["text"]], max_length=config.TASK1_MAX_TEXT_LEN)

with torch.no_grad():
    logits, _ = model3(
        text_batch["input_ids"].to(device), text_batch["attention_mask"].to(device),
        graph_batch.x, graph_batch.edge_index, graph_batch.batch,
    )
    probs3 = torch.sigmoid(logits)[0]
predicted3 = [vocab34[i] for i in (probs3 > 0.5).nonzero().flatten().tolist()]

print("Caption:", example3["text"])
print("True tags:", example3["tags"])
print("Predicted tags (concat fusion):", predicted3)

## 4. Task 4 -- retrieve top-3 clips for a caption

In [ ]:
from task4_contrastive import DualEncoder

model4 = DualEncoder(
    bert_model_name=config.TASK1_MODEL_NAME, gnn_in_channels=config.GRAPH_IN_CHANNELS,
    embed_dim=config.TASK4_EMBED_DIM,
).to(device)
model4.load_state_dict(torch.load(config.CHECKPOINT_DIR / "task4_contrastive_best.pt", map_location=device))
model4.eval()

# Encode every test clip's audio graph once.
graphs = [torch.load(r["graph_path"], weights_only=False) for r in test34]
graph_batch_all = PyGBatch.from_data_list(graphs).to(device)
with torch.no_grad():
    g_all = model4.encode_graph(graph_batch_all.x, graph_batch_all.edge_index, graph_batch_all.batch)

query_caption = test34[0]["text"]
text_batch = model4.bert.tokenize([query_caption], max_length=config.TASK1_MAX_TEXT_LEN)
with torch.no_grad():
    t_query = model4.encode_text(text_batch["input_ids"].to(device), text_batch["attention_mask"].to(device))

similarity = (t_query @ g_all.t())[0]
top3 = similarity.topk(3).indices.tolist()

print("Query caption:", query_caption)
print("Correct track_id:", test34[0]["track_id"])
print("Top-3 retrieved track_ids:", [test34[i]["track_id"] for i in top3])